# 任務 7：驗證分析與敏感度
目標：比較不同處理方案，檢查樣本數及資料生成順序。

**學生版**｜與教師版任務及小節編號一一對應

## 開始之前
**姓名：**何韋仁　**學號／組別：**C114118243　**完成日期：**2026/11/13

本教材可以在 Colab 或 Jupyter 執行，請依序執行儲存格。每份任務的準備儲存格會在記憶體重新建立相同的模擬資料，不會覆寫 CSV。若要分析自己先前產生的 CSV，先上傳檔案，執行準備儲存格後再執行 `df = pd.read_csv("student_learning_data.csv")`；不同資料的驗證數字可能不同。

成績是人為公式生成，資料品質問題在計分後加入。結果只描述模擬資料，不能推論真實學生或證明因果。每個任務請先預測，執行後修改與驗證，最後用自己的話解釋。

### 學生操作方式
1. 先執行下一個完整的資料準備儲存格。每份任務可獨立執行；重啟環境後請由上而下重新執行。
2. 在每個小節的程式中，將 `___` 替換成符合題意的程式碼，並把 `# 學生註解` 改成自己的繁體中文解說（至少說明指定程式的輸入／處理／輸出）。`___` 是刻意留下的空格，完成前執行會報錯。
3. 先記錄預測，執行後完成驗證與解釋；再完成延伸練習。遇到困難可以使用小節專屬 AI 提示詞，仍須檢查、修改、執行與理解。
4. 在文字儲存格中填寫答案，保留程式、執行結果、圖表和 AI 使用紀錄。依教師指示交付已完成的 `.ipynb`。首次執行任務 1 的存檔程式會覆寫同名 practice_student_data.csv，請先確認或改檔名。


In [ ]:
# 載入分析與繪圖套件；np、pd、plt 是慣用縮寫
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 固定種子及呼叫順序，使本課範例能重現
rng = np.random.default_rng(42)
n = 60  # 模擬學生人數
departments = ["資訊管理", "企業管理", "金融管理"]
# 字典每個鍵是欄位名稱，每個值提供 60 位學生的資料
data = {
    "student_id": range(1, n + 1),
    "department": rng.choice(departments, n),
    "sleep_hours": rng.normal(6.7, 1.1, n).round(1),
    "screen_time_hours": rng.normal(5.0, 1.5, n).clip(1, 10).round(1),
    "study_hours": rng.normal(2.5, 1.0, n).clip(0.5, 6).round(1),
    "attendance_rate": rng.normal(88, 8, n).clip(60, 100).round(0),
}
df = pd.DataFrame(data)  # 把字典轉成資料表
noise = rng.normal(0, 5, n)  # 模擬其他未觀察的差異
score = (50 + 3 * df["study_hours"] + 2 * df["sleep_hours"]
         - 1.2 * df["screen_time_hours"] + 0.15 * df["attendance_rate"] + noise)
df["quiz_score"] = score.clip(0, 100).round(0)  # 限制 0 到 100 分
df.loc[[5, 17], "sleep_hours"] = np.nan  # 成績算完後才加入缺失
df.loc[8, "screen_time_hours"] = 15.0  # 成績算完後才加入異常值
print("資料已準備：", df.shape)

## 7.1 定義比較方案
保留 15 小時，或排除超過 10 小時，分別計算螢幕時間與成績的相關係數。此處不假設任一方案一定正確。
**預測：** 去除一個極端值後，係數會不會改變？

### 學生練習 7.1：完成程式與註解

- **程式填空：** 補上保留螢幕時間不超過 10 小時的條件。
- **註解任務：** 將程式中的兩處「學生註解」改寫為自己的說明；第一處解釋填空程式的輸入、處理與輸出，第二處解釋另一行程式的作用。
- **先預測：** 執行前，寫下你預期會看到什麼，或描述操作的效果。

**我的預測：**（請填寫）

**AI 求助提示詞（可直接複製）：**
> 我是 Python 初學者，正在完成資料分析任務 7.1。我的資料表叫 df，請教我如何補上保留螢幕時間不超過 10 小時的條件。 請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 `___` 的一小段程式；不要假設我的執行結果。請提供一個核對方法，並提醒我如何用自己的話撰寫註解。


In [ ]:
# 學生練習 7.1：排除異常值比較方案
# 學生註解 1：這段程式的輸入是原始 DataFrame `df`。處理方式是藉由比較運算 `df["screen_time_hours"] <= 10` 產生布林遮罩 `condition`，並篩選出 True 的行存入 `filtered`。最後輸出排除異常值後的有效配對資料表 `filtered_pairs`。
all_pairs = df.dropna(subset=["screen_time_hours", "quiz_score"])
condition = df["screen_time_hours"] <= 10
filtered = df[condition]
filtered_pairs = filtered.dropna(subset=["screen_time_hours", "quiz_score"])

print("保留筆數：", len(all_pairs), "排除後：", len(filtered_pairs))

# 學生註解 2：這行程式 `df.dropna(subset=["screen_time_hours", "quiz_score"])` 的作用是確保在計算與比較相關係數前，將可能含有缺失值（NaN）的任何資料列徹底剔除，提供一個乾淨無缺失值干擾的基準配對集 `all_pairs`。

# 核對與驗證
assert len(all_pairs) == 60, "⚠️ 原始含異常值之配對數應為 60 筆"
assert len(filtered_pairs) == 59, "⚠️ 排除大於 10 小時的異常值後，有效配對數應為 59 筆"

### 驗證、解釋與延伸 7.1

**驗證與解釋：** 兩種方案各有多少有效配對？差多少筆？

**我的結果與解釋（引用實際輸出）：**（請填寫）

**延伸練習：** 列出不符合條件的 student_id 和螢幕時間。 請在下一個程式儲存格完成，並加上自己的註解。

**延伸練習的預測與驗證：**（請填寫）

**AI 求助提示詞：**
> 請引導我完成這個延伸練習：列出不符合條件的 student_id 和螢幕時間。 目前資料表名稱為 df。請分步解釋程式，建議我如何檢查輸出，不要捏造結果；如果需要使用前面建立的變數，請指出其前置步驟。


In [ ]:
# 延伸練習 7.1：篩選並列出不符合小於等於 10 小時條件的學生學號與數值

# 學生註解 1：這段程式的輸入是原始 DataFrame `df`。處理方式是使用 `~condition`（即 df["screen_time_hours"] > 10）篩選出不合規的列，並僅取出 "student_id" 與 "screen_time_hours" 兩欄，最後輸出這些異常學生的詳細對照資料。
outliers_df = df[~condition][["student_id", "screen_time_hours"]]

display(outliers_df)

# 學生註解 2：此處呼叫 `tolist()` 方法可以將篩選出來的學號轉為 Python 清單，方便我們以簡單的文字格式列印，並直接用於 assert 核對，確保排除對象無誤。
excluded_ids = outliers_df["student_id"].tolist()
print("被排除的學生學號名單：", excluded_ids)

# 驗證檢查
assert excluded_ids == [9], "⚠️ 驗證：被排除的學號應該只有 9 號學生！"

## 7.2 比較結果
敏感度分析觀察選擇變動時，結論對該選擇有多敏感；它不是自動證明哪種結果正確。

### 學生練習 7.2：完成程式與註解

- **程式填空：** 補上排除後配對資料的 Pearson 相關係數。
- **註解任務：** 將程式中的兩處「學生註解」改寫為自己的說明；第一處解釋填空程式的輸入、處理與輸出，第二處解釋另一行程式的作用。
- **先預測：** 執行前，寫下你預期會看到什麼，或描述操作的效果。

**我的預測：**（請填寫）

**AI 求助提示詞（可直接複製）：**
> 我是 Python 初學者，正在完成資料分析任務 7.2。我的資料表叫 df，請教我如何補上排除後配對資料的 Pearson 相關係數。 請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 `___` 的一小段程式；不要假設我的執行結果。請提供一個核對方法，並提醒我如何用自己的話撰寫註解。


In [ ]:
# 學生練習 7.2：比較保留與排除異常值下的相關係數

# 學生註解 1：本程式的輸入是排除異常值後的有效配對資料表 `filtered_pairs`，處理方式為調用 `.corr()` 函數計算其 "screen_time_hours" 與 "quiz_score" 之間的 Pearson 相關係數，最終將計算出的相關係數值 (float) 輸出並指派給變數 `corr_filtered`。
corr_all = all_pairs["screen_time_hours"].corr(all_pairs["quiz_score"])
corr_filtered = filtered_pairs["screen_time_hours"].corr(filtered_pairs["quiz_score"])

print("方案 A (保留異常值)：", corr_all)
print("方案 B (排除異常值)：", corr_filtered)
print("差異（排除減保留）：", corr_filtered - corr_all)

# 學生註解 2：此行 `corr_filtered - corr_all` 的運算作用在於衡量排除異常值後相關係數的變化量（敏感度分析）。若結果為負值，證實排除異常值能讓「螢幕時間」與「學期成績」之間的負相關關係變得更加強烈（負得更多）。

# 自我核對與驗證
assert -1 <= corr_all <= 1, "⚠️ 相關係數必須介於 -1 與 1 之間！"
assert -1 <= corr_filtered <= 1, "⚠️ 相關係數必須介於 -1 與 1 之間！"
assert corr_filtered < corr_all, "💡 驗證：排除異常點 15.0 小時後，負相關強度通常應增強（即相關係數變得更負）。"


### 驗證、解釋與延伸 7.2

**驗證與解釋：** 記錄兩個係數與差值，方向及大小是否改變？此比較能否決定哪個方案必定正確？

**我的結果與解釋（引用實際輸出）：**（請填寫）

**延伸練習：** 比較兩方案 screen_time_hours 的平均值與筆數。 請在下一個程式儲存格完成，並加上自己的註解。

**延伸練習的預測與驗證：**（請填寫）

**AI 求助提示詞：**
> 請引導我完成這個延伸練習：比較兩方案 screen_time_hours 的平均值與筆數。 目前資料表名稱為 df。請分步解釋程式，建議我如何檢查輸出，不要捏造結果；如果需要使用前面建立的變數，請指出其前置步驟。


In [ ]:
# 延伸練習 7.2：比較兩方案 screen_time_hours 的平均值與筆數

# 學生註解 1：本程式輸入為原始配對 `all_pairs` 與篩選後的 `filtered_pairs` 兩個 DataFrame。處理方式為分別取出 "screen_time_hours" 欄位，並調用 .count() 計算有效筆數、調用 .mean() 計算算術平均數，最終輸出兩個方案的筆數與平均值。
screen_count_all = all_pairs["screen_time_hours"].count()
screen_mean_all = all_pairs["screen_time_hours"].mean()

screen_count_filtered = filtered_pairs["screen_time_hours"].count()
screen_mean_filtered = filtered_pairs["screen_time_hours"].mean()

print(f"方案 A (保留異常值) - 筆數: {screen_count_all} 筆, 平均螢幕時間: {screen_mean_all:.4f} 小時")
print(f"方案 B (排除異常值) - 筆數: {screen_count_filtered} 筆, 平均螢幕時間: {screen_mean_filtered:.4f} 小時")
print(f"平均值下降了: {screen_mean_all - screen_mean_filtered:.4f} 小時")

# 學生註解 2：此處利用 assert 斷言來進行結構核對。一來驗證筆數確實差 1 筆，二來核對排除異常大值 15.0 後，方案 B 的平均螢幕時間確實下降，確保資料流與運算邏輯正確無誤。
assert screen_count_all == 60, "⚠️ 方案 A 的資料筆數應為 60 筆！"
assert screen_count_filtered == 59, "⚠️ 方案 B 的資料筆數應為 59 筆！"
assert screen_mean_filtered < screen_mean_all, "💡 驗證成功：排除極端高值 15.0 小時後，平均螢幕時間如預期出現下降。"

## 7.3 資料處理順序
成績先依原始數值計算，才把睡眠設為缺失與螢幕時間改為 15。因此當前顯示值不必然是當初計分的值。

### 學生練習 7.3：完成程式與註解

- **程式填空：** 補上選取 student_id 為 6、9、18 的布林條件。
- **註解任務：** 將程式中的兩處「學生註解」改寫為自己的說明；第一處解釋填空程式的輸入、處理與輸出，第二處解釋另一行程式的作用。
- **先預測：** 執行前，寫下你預期會看到什麼，或描述操作的效果。

**我的預測：**（請填寫）

**AI 求助提示詞（可直接複製）：**
> 我是 Python 初學者，正在完成資料分析任務 7.3。我的資料表叫 df，請教我如何補上選取 student_id 為 6、9、18 的布林條件。 請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 `___` 的一小段程式；不要假設我的執行結果。請提供一個核對方法，並提醒我如何用自己的話撰寫註解。


In [ ]:
# 學生練習 7.3：篩選特定受影響學生名單

# 學生註解 1：本填空程式的輸入為原始資料表 `df`，處理方式是透過 `.isin([6, 9, 18])` 方法判斷學號是否為目標對象，產生布林遮罩 `affected`。最後輸出僅包含 6、9、18 號同學的特定學術與生活作息欄位子集。
affected = df["student_id"].isin([6, 9, 18])
print("--- 受異常與缺失影響的學生對照表 ---")
display(df.loc[affected, ["student_id", "sleep_hours", "screen_time_hours", "quiz_score"]])

# 學生註解 2：此處 `df.loc[affected]` 配合特定欄位串列，能讓我們精確挑選出符合布林遮罩為 True 的資料列，並僅呈現我們關心的學號、睡眠時數、螢幕使用時間與期末成績，避免無關欄位干擾視覺檢視。

# 自我核對與驗證
selected_ids = df.loc[affected, "student_id"].tolist()
assert set(selected_ids) == {6, 9, 18}, "⚠️ 篩選結果的學號不正確，應剛好包含 6, 9, 18！"
print("\n✅ 核對結果：篩選出的學號名單與預期完全相符！")

### 驗證、解釋與延伸 7.3

**驗證與解釋：** 列出的三筆資料哪裡被修改？為何目前的數值不一定是當初計分值？

**我的結果與解釋（引用實際輸出）：**（請填寫）

**延伸練習：** 只選 student_id 為 9 的資料並顯示螢幕時間與成績。 請在下一個程式儲存格完成，並加上自己的註解。

**延伸練習的預測與驗證：**（請填寫）

**AI 求助提示詞：**
> 請引導我完成這個延伸練習：只選 student_id 為 9 的資料並顯示螢幕時間與成績。 目前資料表名稱為 df。請分步解釋程式，建議我如何檢查輸出，不要捏造結果；如果需要使用前面建立的變數，請指出其前置步驟。


In [ ]:
# 延伸練習 7.3：篩選學號 9 的學生並顯示其螢幕時間與成績
# 前置步驟提示：此練習將直接讀取先前定義好的原始資料表 df

# 1. 建立篩選條件：尋找學號（student_id）剛好等於 9 的資料行
# 學生註解 1：這段程式的輸入是資料表 df。處理方式是先透過比較運算 `df["student_id"] == 9` 產生一個僅有一列為 True 的布林 Series（遮罩），並利用 .loc 精確定位該資料行，最後輸出僅包含指定的 "student_id"、"screen_time_hours" 與 "quiz_score" 三個欄位的子資料表。
student_9_df = df.loc[df["student_id"] == 9, ["student_id", "screen_time_hours", "quiz_score"]]

# 2. 呈現篩選結果
display(student_9_df)

# 學生註解 2：此行 `len(student_9_df)` 程式碼的作用是統計並核對篩選出來的資料筆數。由於學號（student_id）在資料表中是唯一的識別碼，因此預期結果應剛好只有 1 筆紀錄，以此驗證篩選條件的精準度。
print(f"篩選出的學生筆數：{len(student_9_df)} 筆")

# 自動化自我核對
assert len(student_9_df) == 1, "⚠️ 驗證失敗：應該只有一筆學號為 9 的資料！"
assert student_9_df["screen_time_hours"].values[0] == 15.0, "⚠️ 驗證失敗：學號 9 的螢幕時間應為 15.0！"
print(
    f"\n✅ 自我核對成功：成功篩選出學號 9 的同學，其螢幕時間確實為 {student_9_df['screen_time_hours'].values[0]} 小時，成績為 {student_9_df['quiz_score'].values[0]} 分！"
)

## 任務交付與 AI 使用紀錄

完成本任務的三個小節，儲存含輸出的 Notebook，依教師指定的方式繳交。任務 8 另需完成 8.3 的完整分析報告及其佐證圖表。

- 使用的 AI 提示詞（未使用請填「未使用」）：
- AI 建議中我修改或拒絕的一處及原因：
- 遇到的錯誤或疑問、修正與驗證方式：
- 本任務一項有數據支持的發現及對應儲存格：
- 一項資料或推論限制：

**交付前檢核：** 已填完程式空格與註解；已填寫預測、驗證、解釋與延伸；已由上到下執行並確認圖表和數字與文字一致。
